# Kisan Mitra — Train on Crop AND Non-Crop Data

One notebook, two models, both drop into `backend/models/`:

| Part | Data | Output |
|---|---|---|
| **A. Disease classifier** | Kaggle *Paddy Doctor* (rice leaf photos, 10 classes) | `paddy_disease_model_fold0.tflite` + `model_meta.json` |
| **B. Crop gate** ("is this a rice leaf?") | Rice photos = positives. **Non-crop** = textures, everyday objects, animals, and *other crops' leaves* (PlantVillage) = negatives | `crop_gate.tflite` + `crop_gate.json` |

Why both: the disease model only knows 10 rice classes, so it will label a shoe "Brown Spot". The gate rejects
non-rice photos before the disease model runs. Part B uses the same recipe as `scripts/train_crop_gate.py`.

**Runtime:** GPU (Runtime -> Change runtime type -> T4/L4/A100). Set `TRAIN_DISEASE = False` to only retrain the gate (~10 min).
Progress is saved to Google Drive; re-run all cells after a disconnect and finished parts are skipped.
You need a Kaggle API token and must have accepted the
[competition rules](https://www.kaggle.com/competitions/paddy-disease-classification/rules).

## 1. Setup and configuration

In [ ]:
!pip install -q kagglehub pyarrow huggingface_hub

In [ ]:
import gc, json, os, random, shutil
from concurrent.futures import ThreadPoolExecutor
from datetime import date

import keras
import numpy as np
import pandas as pd
import tensorflow as tf
from keras import layers
from PIL import Image, ImageOps
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

GPUS = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPU:", GPUS or "none - training will be very slow")

# ---- what to run ----
TRAIN_DISEASE = False       # Part A (set True to also train the disease model)
TRAIN_GATE = True           # Part B

# ---- Part A: disease classifier ----
BACKBONE = "efficientnetv2s"   # "efficientnetv2s" | "efficientnetv2b3" | "convnext_tiny"
IMG_H, IMG_W = 384, 288        # Paddy Doctor photos are 3:4 portrait
CACHE_H, CACHE_W = 512, 384
BATCH_SIZE = 32
HEAD_EPOCHS, EPOCHS = 2, 15
PEAK_LR, WEIGHT_DECAY, LABEL_SMOOTHING = 3e-4, 0.05, 0.1
VAL_FRACTION = 0.2
TARGET_ACCEPTED_ACCURACY = 0.97
QUANTIZE = "dynamic"           # "dynamic" | "float16" | "none"

# ---- Part B: crop gate ----
GATE_SIZE = 224
POS_PER_CLASS = 250            # rice photos used per disease class
NEG_PER_SOURCE = 2500          # non-crop images per source dataset
TARGET_RECALL = 0.98           # share of real rice photos the gate must let through
# (HF dataset repo, parquet files or None for all, max images). PlantVillage = other crops = hardest negatives.
NEG_SOURCES = [
    ("tanganke/dtd", None, NEG_PER_SOURCE),
    ("tanganke/stl10", ["data/train-00000-of-00001.parquet"], NEG_PER_SOURCE),
    ("dpdl-benchmark/caltech101", ["data/train-00000-of-00001.parquet"], NEG_PER_SOURCE),
    ("GVJahnavi/PlantVillage_dataset", ["data/test-00000-of-00001.parquet"], 3000),
]
EXTRA_NON_CROP_DIR = "/content/drive/MyDrive/kisan_mitra/not_crop"   # optional: your own non-rice photos (any subfolders)

MIXED_PRECISION = bool(GPUS) and TRAIN_DISEASE   # the gate trains/exports in plain float32
SEED = 42
WORK_DIR = "/content/kisan_mitra"
SAVE_DIR = "/content/drive/MyDrive/kisan_mitra/model_combined"
DISPLAY_NAMES = {   # must match backend/diseases.json
    "bacterial_leaf_blight": "Bacterial Leaf Blight", "bacterial_leaf_streak": "Bacterial Leaf Streak",
    "bacterial_panicle_blight": "Bacterial Panicle Blight", "blast": "Blast", "brown_spot": "Brown Spot",
    "dead_heart": "Dead Heart", "downy_mildew": "Downy Mildew", "hispa": "Hispa",
    "normal": "Normal", "tungro": "Tungro",
}
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    SAVE_DIR = os.path.join(WORK_DIR, "saved"); EXTRA_NON_CROP_DIR = ""
    print("Not in Colab - saving to", SAVE_DIR)
os.makedirs(WORK_DIR, exist_ok=True); os.makedirs(SAVE_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); keras.utils.set_random_seed(SEED)
keras.mixed_precision.set_dtype_policy("mixed_float16" if MIXED_PRECISION else "float32")

## 2. Download the crop data (Paddy Doctor)

In [ ]:
import getpass, kagglehub
token = getpass.getpass("Paste your Kaggle API token: ")
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/access_token"), "w") as f: f.write(token.strip())
os.chmod(os.path.expanduser("~/.kaggle/access_token"), 0o600)
DATA_DIR = kagglehub.competition_download("paddy-disease-classification")

CLASS_FOLDERS = sorted(DISPLAY_NAMES)
CLASS_NAMES = [DISPLAY_NAMES[c] for c in CLASS_FOLDERS]
NUM_CLASSES = len(CLASS_FOLDERS)
df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
assert set(df.label) == set(CLASS_FOLDERS), set(df.label) ^ set(CLASS_FOLDERS)
df["src"] = [os.path.join(DATA_DIR, "train_images", l, i) for l, i in zip(df.label, df.image_id)]
df["y"] = df.label.map({c: i for i, c in enumerate(CLASS_FOLDERS)})
print(len(df), "labelled rice images"); print(df.label.value_counts())

def prepare_image(img):
    """Same as backend/inference.py: EXIF rotation, landscape -> portrait."""
    img = ImageOps.exif_transpose(img).convert("RGB")
    if img.width > img.height: img = img.rotate(90, expand=True)
    return img

## 3. Download the non-crop data

Textures, everyday objects/animals, and **other crops' leaves** (PlantVillage). Your own non-rice photos in
`EXTRA_NON_CROP_DIR` on Drive are added too — put the kinds of wrong photos farmers actually take (hands, soil, selfies, rice *grain*, weeds).

In [ ]:
import io, pyarrow.parquet as pq
from huggingface_hub import hf_hub_download, list_repo_files
NEG_DIR = os.path.join(WORK_DIR, "not_crop")

def image_column(table):
    for n in table.column_names:
        if n in ("image", "img"): return n
    raise KeyError(table.column_names)

def save_set(repo, files, cap):
    name = repo.split("/")[-1]; dest = os.path.join(NEG_DIR, name)
    if os.path.isdir(dest) and len(os.listdir(dest)) >= min(cap, 500):
        print(name, ": already there"); return
    os.makedirs(dest, exist_ok=True)
    files = files or [f for f in list_repo_files(repo, repo_type="dataset") if f.endswith(".parquet")]
    random.shuffle(files); saved = 0
    for f in files:
        pf = pq.ParquetFile(hf_hub_download(repo, f, repo_type="dataset"))
        col = image_column(pf.schema_arrow.empty_table())
        per_file = max(1, (cap - saved) // max(1, len(files)))
        for batch in pf.iter_batches(batch_size=256, columns=[col]):
            for cell in batch.column(0).to_pylist():
                if saved >= cap or per_file <= 0: break
                if random.random() > 0.5 and len(files) > 1: continue
                try:
                    Image.open(io.BytesIO(cell["bytes"])).convert("RGB").save(os.path.join(dest, f"{saved:05d}.jpg"), quality=90)
                    saved += 1; per_file -= 1
                except Exception: continue
            if saved >= cap or per_file <= 0: break
        if saved >= cap: break
    print(name, ": saved", saved)

for repo, files, cap in NEG_SOURCES:
    try: save_set(repo, files, cap)
    except Exception as exc: print(repo, "FAILED", repr(exc))   # one dead source shouldn't stop the rest

def list_images(folder):
    return [os.path.join(d, f) for d, _, fs in os.walk(folder) for f in fs if f.lower().endswith(IMG_EXT)]

neg_files = list_images(NEG_DIR)
if EXTRA_NON_CROP_DIR and os.path.isdir(EXTRA_NON_CROP_DIR):
    extra = list_images(EXTRA_NON_CROP_DIR); neg_files += extra; print("extra non-crop photos from Drive:", len(extra))
print("total non-crop images:", len(neg_files))

# Part A — Disease classifier (crop data)
## 4. Cache images and build the input pipeline

In [ ]:
def cache_image(src, dst):
    if os.path.exists(dst): return
    with Image.open(src) as img:
        img = prepare_image(img).resize((CACHE_W, CACHE_H), Image.Resampling.BILINEAR)
    img.save(dst, quality=95)

if TRAIN_DISEASE:
    cdir = os.path.join(WORK_DIR, "cache"); os.makedirs(cdir, exist_ok=True)
    dsts = [os.path.join(cdir, os.path.basename(s)) for s in df.src]
    with ThreadPoolExecutor(16) as pool: list(pool.map(cache_image, df.src, dsts))
    df["path"] = dsts
    tr_idx, va_idx = train_test_split(df.index, test_size=VAL_FRACTION, stratify=df.y, random_state=SEED)
    train_df, val_df = df.loc[tr_idx], df.loc[va_idx]
    counts = train_df.y.value_counts().sort_index()
    w = (len(train_df) / (NUM_CLASSES * counts)) ** 0.5
    CLASS_WEIGHTS = (w / w.mean()).to_numpy("float32")
    print("train", len(train_df), "| val", len(val_df))

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

def _augment(raw, label, weight):
    img = tf.io.decode_jpeg(raw, channels=3)
    img = tf.cond(tf.random.uniform([]) < 0.3, lambda: tf.image.random_jpeg_quality(img, 40, 95), lambda: img)
    img.set_shape([None, None, 3])
    begin, size, _ = tf.image.sample_distorted_bounding_box(
        tf.shape(img), bounding_boxes=tf.constant([0.0, 0.0, 1.0, 1.0], shape=[1, 1, 4]),
        min_object_covered=0.0, aspect_ratio_range=(0.6, 0.95), area_range=(0.5, 1.0),
        max_attempts=10, use_image_if_no_bounding_boxes=True)
    img = tf.image.resize(tf.slice(img, begin, size), (IMG_H, IMG_W), antialias=True)
    img = tf.image.random_flip_up_down(tf.image.random_flip_left_right(img))
    img = img / 255.0
    img = tf.image.random_brightness(img, 0.15)
    img = tf.image.random_contrast(img, 0.8, 1.2)
    img = tf.image.random_saturation(img, 0.8, 1.2)
    img = tf.image.random_hue(img, 0.03)
    img = tf.clip_by_value(img, 0.0, 1.0) * 255.0
    return img, tf.one_hot(label, NUM_CLASSES), weight

def _load_eval(raw, label):
    img = tf.io.decode_jpeg(raw, channels=3)
    return tf.image.resize(img, (IMG_H, IMG_W), antialias=True), tf.one_hot(label, NUM_CLASSES)

def _read(path, label, weight): return tf.io.read_file(path), label, weight

def make_train_ds(frame):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.y.values.astype("int32"), CLASS_WEIGHTS[frame.y.values]))
    ds = ds.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True).map(_read, num_parallel_calls=AUTOTUNE)
    return ds.map(_augment, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE, drop_remainder=True).prefetch(AUTOTUNE)

def make_eval_ds(frame, labels=True):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.values, frame.y.values.astype("int32")))
    ds = ds.map(lambda p, l: (tf.io.read_file(p), l), num_parallel_calls=AUTOTUNE).map(_load_eval, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)
    return ds if labels else ds.map(lambda x, y: x)

## 5. Train the disease model

In [ ]:
def build_model(weights="imagenet"):
    shape = (IMG_H, IMG_W, 3)
    ctor = {"efficientnetv2s": keras.applications.EfficientNetV2S, "efficientnetv2b3": keras.applications.EfficientNetV2B3,
            "convnext_tiny": keras.applications.ConvNeXtTiny}[BACKBONE]
    base = ctor(include_top=False, weights=weights, input_shape=shape)   # includes preprocessing; takes raw 0-255
    inputs = keras.Input(shape)
    x = base(inputs, training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inputs, layers.Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)), base

def set_base_trainable(base, trainable):
    base.trainable = trainable
    if trainable:
        for l in base.layers:
            if isinstance(l, layers.BatchNormalization): l.trainable = False

def compile_model(model, peak_lr, total_steps, warmup_steps):
    sched = keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=peak_lr * 0.01 if warmup_steps else peak_lr, decay_steps=max(total_steps - warmup_steps, 1),
        alpha=0.01, warmup_target=peak_lr if warmup_steps else None, warmup_steps=warmup_steps)
    opt = keras.optimizers.AdamW(learning_rate=sched, weight_decay=WEIGHT_DECAY, clipnorm=1.0)
    opt.exclude_from_weight_decay(var_names=["bias", "gamma", "beta"])
    model.compile(optimizer=opt, loss=keras.losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING), metrics=["accuracy"])

DISEASE_WEIGHTS = os.path.join(SAVE_DIR, f"{BACKBONE}_disease.weights.h5")
if TRAIN_DISEASE:
    if os.path.exists(DISEASE_WEIGHTS):
        print("Disease model already trained - skipping", DISEASE_WEIGHTS)
    else:
        keras.backend.clear_session(); gc.collect()
        train_ds, val_ds = make_train_ds(train_df), make_eval_ds(val_df)
        steps = len(train_df) // BATCH_SIZE
        model, base = build_model()
        set_base_trainable(base, False); compile_model(model, 1e-3, steps * HEAD_EPOCHS, 0)
        model.fit(train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS, verbose=2)
        set_base_trainable(base, True); compile_model(model, PEAK_LR, steps * EPOCHS, steps)
        model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, verbose=2)
        model.save_weights(DISEASE_WEIGHTS)

## 6. Evaluate, calibrate, export disease model

In [ ]:
def calibrate(probs, T):  # also used by the test cell
    z = np.log(np.clip(probs, 1e-12, 1.0)) / T; z -= z.max(1, keepdims=True); e = np.exp(z)
    return e / e.sum(1, keepdims=True)

if TRAIN_DISEASE:
    keras.backend.clear_session()
    model, _ = build_model(weights=None); model.load_weights(DISEASE_WEIGHTS)
    ds = make_eval_ds(val_df, labels=False)
    probs = (model.predict(ds, verbose=0) + model.predict(ds.map(lambda x: tf.reverse(x, axis=[2])), verbose=0)) / 2
    y_true = val_df.y.values
    print(classification_report(y_true, probs.argmax(1), target_names=CLASS_NAMES, digits=3, zero_division=0))

    def nll(T): return -np.log(np.clip(calibrate(probs, T)[np.arange(len(y_true)), y_true], 1e-12, 1)).mean()
    TEMPERATURE = float(minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded").x)
    cal = calibrate(probs, TEMPERATURE); conf, correct = cal.max(1), cal.argmax(1) == y_true
    RECOMMENDED_THRESHOLD = 90.0
    for t in range(30, 100, 5):
        acc = conf >= t / 100
        if acc.any() and correct[acc].mean() >= TARGET_ACCEPTED_ACCURACY: RECOMMENDED_THRESHOLD = float(t); break
    DISEASE_ACC = float((probs.argmax(1) == y_true).mean())
    print(f"accuracy {DISEASE_ACC:.4f} | temperature {TEMPERATURE:.3f} | recommended threshold {RECOMMENDED_THRESHOLD:.0f}%")

In [ ]:
OUT_DIR = os.path.join(WORK_DIR, "tflite"); shutil.rmtree(OUT_DIR, ignore_errors=True); os.makedirs(OUT_DIR)
if TRAIN_DISEASE:
    keras.mixed_precision.set_dtype_policy("float32")
    keras.backend.clear_session()
    model, _ = build_model(weights=None); model.load_weights(DISEASE_WEIGHTS)
    model.export(os.path.join(WORK_DIR, "savedmodel_disease"))
    conv = tf.lite.TFLiteConverter.from_saved_model(os.path.join(WORK_DIR, "savedmodel_disease"))
    if QUANTIZE == "dynamic": conv.optimizations = [tf.lite.Optimize.DEFAULT]
    elif QUANTIZE == "float16": conv.optimizations = [tf.lite.Optimize.DEFAULT]; conv.target_spec.supported_types = [tf.float16]
    name = "paddy_disease_model_fold0.tflite"
    open(os.path.join(OUT_DIR, name), "wb").write(conv.convert())
    json.dump({"models": [name], "class_names": CLASS_NAMES, "backbone": BACKBONE, "input_height": IMG_H, "input_width": IMG_W,
               "preprocess": "none", "tta": "hflip", "temperature": round(TEMPERATURE, 4),
               "recommended_threshold": RECOMMENDED_THRESHOLD, "oof_accuracy": round(DISEASE_ACC, 4),
               "quantize": QUANTIZE, "pseudo_label": False, "trained_on": str(date.today())},
              open(os.path.join(OUT_DIR, "model_meta.json"), "w"), indent=2)
    print(name, round(os.path.getsize(os.path.join(OUT_DIR, name)) / 1e6, 1), "MB")

# Part B — Crop gate (crop vs non-crop)
## 7. Train the gate

Positives: rice photos from every disease class. Negatives: the non-crop images above plus synthetic blank / dark / noise frames.
MobileNetV2 (ImageNet) features -> logistic regression; the head is baked into the `.tflite`, which outputs P(rice leaf).
The split is by source photo, so crops of one image never land on both sides.

In [ ]:
def views(img, n=2):
    """Full frame plus randomly zoomed/flipped crops - farmers shoot both wide and close."""
    img = prepare_image(img); S = GATE_SIZE
    out = [np.asarray(img.resize((S, S), Image.Resampling.BILINEAR), dtype=np.float32)]
    for _ in range(n - 1):
        sc = random.uniform(0.55, 1.0); w, h = int(img.width * sc), int(img.height * sc)
        x, y = random.randint(0, img.width - w), random.randint(0, img.height - h)
        c = img.crop((x, y, x + w, y + h))
        if random.random() < 0.5: c = ImageOps.mirror(c)
        out.append(np.asarray(c.resize((S, S), Image.Resampling.BILINEAR), dtype=np.float32))
    return out

def synthetic_negatives(n):
    S = GATE_SIZE; out = []
    for i in range(n):
        k = i % 4
        if k == 0: a = np.ones((S, S, 3), np.float32) * np.random.randint(0, 256, 3)
        elif k == 1: a = np.random.rand(S, S, 3).astype(np.float32) * 40
        elif k == 2: a = np.random.rand(S, S, 3).astype(np.float32) * 255
        else:
            c1, c2 = np.random.randint(0, 256, 3), np.random.randint(0, 256, 3)
            t = np.linspace(0, 1, S, dtype=np.float32)[:, None, None]
            a = np.broadcast_to(c1 * (1 - t) + c2 * t, (S, S, 3)).copy()
        out.append(a.astype(np.float32))
    return out

def gate_extractor():
    inp = keras.Input((GATE_SIZE, GATE_SIZE, 3), name="image")   # raw 0-255 RGB
    x = layers.Rescaling(1 / 127.5, offset=-1)(inp)
    base = keras.applications.MobileNetV2(input_shape=(GATE_SIZE, GATE_SIZE, 3), include_top=False, weights="imagenet", pooling="avg")
    return keras.Model(inp, base(x))

def embed(ext, arrays):
    return np.concatenate([ext.predict(np.stack(arrays[i:i + 64]), verbose=0) for i in range(0, len(arrays), 64)])

if TRAIN_GATE:
    pos_files = []
    for c in CLASS_FOLDERS:
        fs = df[df.label == c].src.tolist(); random.shuffle(fs); pos_files += fs[:POS_PER_CLASS]
    random.shuffle(neg_files)
    assert len(neg_files) >= 200, "need more non-crop images"
    print(f"positives {len(pos_files)} | negatives {len(neg_files)}")
    ext = gate_extractor()
    pos_tr, pos_va = train_test_split(pos_files, test_size=0.2, random_state=0)
    neg_tr, neg_va = train_test_split(neg_files, test_size=0.2, random_state=0)

    def load(files, label, nv, extra=()):
        """Embed in batches of 64 so only a few images are ever in RAM."""
        feats, buf = [], list(extra)
        def flush():
            nonlocal buf
            if buf: feats.append(ext.predict(np.stack(buf), verbose=0).astype(np.float32)); buf = []
        n = len(buf)
        for f in files:
            try:
                with Image.open(f) as im: v = views(im, nv)
            except Exception: continue
            buf += v; n += len(v)
            if len(buf) >= 64: flush()
        flush()
        return np.concatenate(feats), np.full(n, label)

    sets = {}
    for nm, (pf, nf) in {"train": (pos_tr, neg_tr), "val": (pos_va, neg_va)}.items():
        Xp, yp = load(pf, 1, 2)
        Xn, yn = load(nf, 0, 2, extra=synthetic_negatives(max(40, len(nf) // 12)))
        sets[nm] = (np.concatenate([Xp, Xn]), np.concatenate([yp, yn]))
        print(nm, len(sets[nm][1]), "samples")
    (Xtr, ytr), (Xva, yva) = sets["train"], sets["val"]
    clf = LogisticRegression(C=0.5, max_iter=2000, class_weight="balanced").fit(Xtr, ytr)
    p = clf.predict_proba(Xva)[:, 1]
    auc = roc_auc_score(yva, p); fpr, tpr, thr = roc_curve(yva, p)
    ok = np.where(tpr >= TARGET_RECALL)[0][0]; GATE_THRESHOLD = float(thr[ok])
    print(f"val AUC {auc:.4f} | threshold {GATE_THRESHOLD:.3f} keeps {tpr[ok]:.1%} of rice photos, lets through {fpr[ok]:.1%} of non-rice")

## 8. Export the gate

In [ ]:
if TRAIN_GATE:
    # Export in float32: under mixed_float16 the graph contains ops TFLite can't convert (ERROR_NEEDS_FLEX_OPS).
    old_policy = keras.mixed_precision.dtype_policy()
    keras.mixed_precision.set_dtype_policy("float32")
    keras.backend.clear_session()
    ext_export = gate_extractor()
    ext_export.set_weights(ext.get_weights())

    head = layers.Dense(1, activation="sigmoid", name="rice_leaf")
    gate = keras.Sequential([ext_export, head]); gate.build((None, GATE_SIZE, GATE_SIZE, 3))
    head.set_weights([clf.coef_.T.astype(np.float32), clf.intercept_.astype(np.float32)])
    probe = np.random.rand(1, GATE_SIZE, GATE_SIZE, 3).astype(np.float32) * 255
    ref = float(clf.predict_proba(ext_export.predict(probe, verbose=0))[0, 1])
    assert abs(float(gate.predict(probe, verbose=0)[0, 0]) - ref) < 1e-3, "baked head disagrees with sklearn"

    conv = tf.lite.TFLiteConverter.from_keras_model(gate); conv.optimizations = [tf.lite.Optimize.DEFAULT]
    b = conv.convert(); open(os.path.join(OUT_DIR, "crop_gate.tflite"), "wb").write(b)
    json.dump({"threshold": round(GATE_THRESHOLD, 4), "val_auc": round(float(auc), 4), "rice_recall": round(float(tpr[ok]), 4),
               "non_rice_pass_rate": round(float(fpr[ok]), 4), "input_size": GATE_SIZE},
              open(os.path.join(OUT_DIR, "crop_gate.json"), "w"), indent=2)
    print(f"crop_gate.tflite {len(b) / 1e6:.1f} MB")
    keras.mixed_precision.set_dtype_policy(old_policy)

## 9. Save and download

Copy everything from the zip into the repo's `backend/models/`.

In [ ]:
shutil.copytree(OUT_DIR, os.path.join(SAVE_DIR, "tflite"), dirs_exist_ok=True)
archive = shutil.make_archive(os.path.join(WORK_DIR, "kisan_mitra_models"), "zip", OUT_DIR)
print(os.listdir(OUT_DIR), "->", archive)
from google.colab import files
files.download(archive)

## 10. Try your own photos

Upload rice leaves *and* random things (a shoe, a tomato leaf). The gate should reject the latter; the disease model should name the former.
Needs both parts to have run in this session.

In [ ]:
from google.colab import files
uploaded = files.upload()
gate_it = tf.lite.Interpreter(os.path.join(OUT_DIR, "crop_gate.tflite")); gate_it.allocate_tensors()
gmeta = json.load(open(os.path.join(OUT_DIR, "crop_gate.json")))
have_disease = os.path.exists(os.path.join(OUT_DIR, "paddy_disease_model_fold0.tflite"))
if have_disease:
    dis_it = tf.lite.Interpreter(os.path.join(OUT_DIR, "paddy_disease_model_fold0.tflite")); dis_it.allocate_tensors()
    meta = json.load(open(os.path.join(OUT_DIR, "model_meta.json")))

def run(it, arr):
    it.set_tensor(it.get_input_details()[0]["index"], np.ascontiguousarray(arr[None]))
    it.invoke(); return it.get_tensor(it.get_output_details()[0]["index"])[0]

def load_arr(f, w, h):
    with Image.open(f) as im: return np.asarray(prepare_image(im).resize((w, h), Image.Resampling.BILINEAR), dtype=np.float32)

for f in uploaded:
    pr = float(run(gate_it, load_arr(f, GATE_SIZE, GATE_SIZE))[0])
    if pr < gmeta["threshold"]:
        print(f"{f}: NOT A RICE LEAF (P={pr:.2f})"); continue
    if not have_disease:
        print(f"{f}: rice leaf (P={pr:.2f})"); continue
    a = load_arr(f, meta["input_width"], meta["input_height"])
    probs = calibrate(((run(dis_it, a) + run(dis_it, a[:, ::-1, :])) / 2)[None], meta["temperature"])[0]
    top = probs.argsort()[::-1][:3]
    print(f"{f}: rice leaf (P={pr:.2f}) -> " + ", ".join(f"{CLASS_NAMES[i]} {probs[i]:.1%}" for i in top))